---
# Post-TARE Model Run: Grid Impact Analysis
---

**Author:** Jordan M. Joseph, PhD — Carnegie Mellon University

Computes lectricity demand change and changes in peak demand at the county-level and state-level under various adoption scenarios.

**Prerequisite:** Run the preTARE notebook first (or ensure EUSS data is loaded).

See `README_GRID_IMPACT.md` for methodology notes and design decisions. --> PLACEHOLDER NEEDS CREATED

---
## Step 0: Imports and Configuration
---

In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from config import PROJECT_ROOT
from cmu_tare_model.constants import (
    ALLOWED_HOUSING_TYPES,
    VALID_MENU_MPS,
    VERBOSE,
    REMDB_COST_SCENARIO_KEYS,
    RCM_MODELS,
    PRIVATE_DISCOUNT_RATE_SHORT_KEYS,
)

from cmu_tare_model.utils.column_names import (
    create_npv_col,
    create_capital_col,
)

from cmu_tare_model.utils.load_exported_results_to_df import load_measure_package_data

from cmu_tare_model.adoption_kpis.kpi_functions import (
    mp_to_upgrade,
    load_euss_baseline,
    load_euss_upgrade,
    calculate_price_ratios,
    compute_thermal_cop_by_state,
    compute_spark_gap_metrics,
    compute_scenario_demand,
    aggregate_demand_by_state,
    FUEL_PRICES_PATH,
    SHAPEFILE_PATH,
    HEATING_FUEL_COLS,
    HP_BACKUP_ELEC_COL,
    HP_FANS_PUMPS_COL,
)
from cmu_tare_model.adoption_kpis.visualize_geospatial_data import (
    prepare_state_geodataframe,
    create_choropleth_map,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

print("✓ Imports loaded")

Project root directory: c:\users\jorda\desktop\projects\cmu-tare-model
✓ Imports loaded


---
## Step 0b: Measure Package Selection
---

In [2]:
SELECTABLE_MPS = [mp for mp in VALID_MENU_MPS if mp != 0]

try:
    _ = input_measure_package
    batch_mode = True
    selected_mps = [int(input_measure_package)]
    print(f"BATCH MODE: Running for MP{selected_mps[0]}")
except NameError:
    batch_mode = False
    print(f"Available measure packages: {SELECTABLE_MPS}")
    mp_input = input("Enter MP numbers (comma-separated, or 'all'): ").strip()
    if mp_input.lower() == 'all':
        selected_mps = SELECTABLE_MPS
    else:
        selected_mps = [int(x.strip()) for x in mp_input.split(',') if x.strip().isdigit()]
        selected_mps = [mp for mp in selected_mps if mp in SELECTABLE_MPS]
    if not selected_mps:
        selected_mps = [4]
        print("No valid MPs selected. Defaulting to MP4.")

print(f"\nSelected measure packages: {selected_mps}")

Available measure packages: [3, 4]

Selected measure packages: [3]


---
## Step 0c: Load TARE Model Data (Measure Packages 3, 4)
---

Load pre-computed TARE model outputs for the selected measure packages.
Required for Step 4d (Private NPV extraction).

If the top-section data loading cells have already been run, this will reuse `DATAFRAMES_BY_MP`.

In [3]:
# =============================================================================
# STEP 0c: LOAD TARE MODEL DATA (for NPV extraction)
# =============================================================================
# Check if DATAFRAMES_BY_MP was already loaded by the top-section cells.
# If not, prompt for the output folder and load for selected MPs.

try:
    _ = DATAFRAMES_BY_MP
    print(f"DATAFRAMES_BY_MP already loaded: {list(DATAFRAMES_BY_MP.keys())}")
except NameError:
    print("DATAFRAMES_BY_MP not found — loading TARE model outputs...")

    # Check if output_folder_path is already defined (from top section)
    try:
        _ = output_folder_path
        print(f"  Using existing output_folder_path: {output_folder_path}")
    except NameError:
        output_folder_path = os.path.join(PROJECT_ROOT, "cmu_tare_model", "output_results")
        location_id = input("Enter location ID (e.g., 'National' or 'PA'): ").strip()
        model_run_date_time = input("Enter model run timestamp (YYYY-MM-DD_HH-MM): ").strip()
        print(f"  output_folder_path: {output_folder_path}")
        print(f"  location_id: {location_id}")
        print(f"  model_run_date_time: {model_run_date_time}")

    DATAFRAMES_BY_MP = {}
    for mp in selected_mps:
        DATAFRAMES_BY_MP[mp] = load_measure_package_data(
            mp, output_folder_path, location_id, model_run_date_time
        )

    print(f"\n✓ Loaded TARE data for MPs: {list(DATAFRAMES_BY_MP.keys())}")

DATAFRAMES_BY_MP not found — loading TARE model outputs...
  output_folder_path: c:\users\jorda\desktop\projects\cmu-tare-model\cmu_tare_model\output_results
  location_id: National
  model_run_date_time: 2026-04-10_00-05
Loading MP3 data...
  fixed_base: ✓ 
MP3 loading complete!


✓ Loaded TARE data for MPs: [3]


---
## Step 1: Load EUSS Data (For data validation: EUSS has individual household peak load kW values)
---

In [4]:
print("=" * 80)
print("STEP 1: LOAD EUSS DATA")
print("=" * 80)

df_baseline = load_euss_baseline()
print(f"  Baseline: {len(df_baseline):,} occupied SF homes")

upgrade_data = {}
for mp in selected_mps:
    upgrade_name = mp_to_upgrade(mp)
    print(f"\nLoading MP{mp} ({upgrade_name})...")
    upgrade_data[mp] = load_euss_upgrade(upgrade_name)
    print(f"  MP{mp}: {len(upgrade_data[mp]):,} applicable homes")

print(f"\n✓ STEP 1 COMPLETE")

STEP 1: LOAD EUSS DATA
Loading baseline from: c:\users\jorda\desktop\projects\cmu-tare-model\cmu_tare_model\data\euss_data\resstock_amy2018_release_1.1\national\csv\baseline_metadata_and_annual_results.csv
  After occupancy filter: 482,597 / 548,916
  After housing type filter (['Single-Family Attached', 'Single-Family Detached']): 331,531
  Baseline: 331,531 occupied SF homes

Loading MP3 (upgrade03)...
Loading upgrade03 from: c:\users\jorda\desktop\projects\cmu-tare-model\cmu_tare_model\data\euss_data\resstock_amy2018_release_1.1\national\csv\upgrade03_metadata_and_annual_results.csv
  After occupancy filter: 482,597 / 548,916
  After housing type filter: 331,531
  After applicability filter: 331,526
  MP3: 331,526 applicable homes

✓ STEP 1 COMPLETE


# Export a dictionary of the building IDs (index of dfs) where the households are potential adopters

In [ ]:
df_mp3 = upgrade_data[3]
df_mp3

,weight,applicability,in.ashrae_iecc_climate_zone_2004,in.county,in.geometry_building_type_recs,in.heating_fuel,in.hvac_heating_efficiency,in.hvac_heating_type_and_fuel,in.state,in.vacancy_status,out.electricity.heating_fans_pumps.energy_consumption.kwh,out.electricity.heating_hp_bkup.energy_consumption.kwh,out.electricity.heating.energy_consumption.kwh,out.fuel_oil.heating.energy_consumption.kwh,out.natural_gas.heating.energy_consumption.kwh,out.propane.heating.energy_consumption.kwh,out.load.heating.energy_delivered.kbtu
bldg_id,,,,,,,,,,,,,,,,,
1,242.131013,True,4A,G5100330,Single-Family Detached,Propane,"Fuel Furnace, 76% AFUE",Propane Fuel Furnace,VA,Occupied,261.126323,283.692796,4636.970470,0.0,0.0,0.0,44278.0
2,242.131013,True,6B,G5600250,Single-Family Detached,Natural Gas,"Fuel Furnace, 80% AFUE",Natural Gas Fuel Furnace,WY,Occupied,939.292779,11116.478756,9174.296775,0.0,0.0,0.0,132946.0
5,242.131013,True,4A,G2901690,Single-Family Detached,Electricity,"Electric Furnace, 100% AFUE",Electricity Electric Furnace,MO,Occupied,538.078485,2256.940310,5494.203349,0.0,0.0,0.0,61868.0
6,242.131013,True,2A,G1200990,Single-Family Detached,NaN,NaN,NaN,FL,Occupied,0.293071,0.000000,12.308985,0.0,0.0,0.0,158.0
8,242.131013,True,5A,G3101470,Single-Family Detached,Natural Gas,"Fuel Furnace, 76% AFUE",Natural Gas Fuel Furnace,NE,Occupied,421.143128,3311.996162,6788.991337,0.0,0.0,0.0,74899.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549993,242.131013,True,3C,G0600010,Single-Family Detached,Natural Gas,"Fuel Wall/Floor Furnace, 68% AFUE",Natural Gas Fuel Wall/Floor Furnace,CA,Occupied,27.255610,56.269645,1776.303755,0.0,0.0,0.0,24138.0
549994,242.131013,True,4A,G1100010,Single-Family Attached,Natural Gas,"Fuel Furnace, 76% AFUE",Natural Gas Fuel Furnace,DC,Occupied,104.333301,236.801425,1640.025708,0.0,0.0,0.0,16653.0
549995,242.131013,True,2A,G4804890,Single-Family Detached,Electricity,"Electric Furnace, 100% AFUE",Electricity Electric Furnace,TX,Occupied,16.705051,5.275279,189.910053,0.0,0.0,0.0,1601.0


: 

# Build Stock Query (BSQ) Setup and Tests

# Aggregate Peak Demand for 100% adoption

This initial test does not require the adopter building IDs. This will involve county-level aggregates. Most reproducible in two methods:
1. County specific peak demand values agnostic of other counties or the nation (neighboring counties could have different timestamps for the peak)
2. Choose the coldest day of the year or day in which the peak demand is highest. Use the same timestamp for all counties

# Aggregate Peak Demand for Adoption Scenarios

This involves the steps outlined above but with custom aggregations. Non-adopters use the existing baseline demand profile and adopters use the post-retrofit demand profile (e.g., MP3 and MP4)

# Validate Results with EUSS peak load values
https://data.openei.org/s3_viewer?bucket=oedi-data-lake&prefix=nrel-pds-building-stock%2Fend-use-load-profiles-for-us-building-stock%2F2022%2Fresstock_amy2018_release_1.1%2F

EUSS (ResStock 2022.1) has inidvidual household peak load values assigned. 

This initial test does not require the adopter building IDs. This will involve county-level aggregates. Most reproducible in two methods:
1. County specific peak demand values agnostic of other counties or the nation (neighboring counties could have different timestamps for the peak)
2. Choose the coldest day of the year or day in which the peak demand is highest. Use the same timestamp for all counties

---
## Step X: Demand Change Under Adoption Scenario
---

Two metrics: **electricity demand change** (grid impact) and **site energy change** (efficiency).

In [ ]:
print(f"===== STEP Xa: SCENARIO DEMAND (MP{primary_mp}, 100% adoption, all fuels) =====")
df_demand = compute_scenario_demand(df_baseline, df_upgrade_primary, fuel_filter=None, verbose=True)

print(f"\n--- Sample: gas homes ---")
gas_sample = df_demand[df_demand['in.heating_fuel'] == 'Natural Gas'].head(3)
print(gas_sample[['in.state', 'in.heating_fuel', 'baseline_electric_kwh',
                   'baseline_heating_total_kwh', 'retrofit_electric_kwh',
                   'elec_demand_change_kwh', 'site_energy_change_kwh']].to_string())

print(f"\n--- Sample: electric baseboard homes ---")
elec_sample = df_demand[df_demand['in.heating_fuel'] == 'Electricity'].head(3)
print(elec_sample[['in.state', 'in.heating_fuel', 'baseline_electric_kwh',
                    'baseline_heating_total_kwh', 'retrofit_electric_kwh',
                    'elec_demand_change_kwh', 'site_energy_change_kwh']].to_string())
print("\n✓ STEP Xa COMPLETE")

In [ ]:
print("===== STEP Xb: AGGREGATE DEMAND BY STATE =====")
df_demand_state = aggregate_demand_by_state(df_demand, verbose=True)

print(f"\n--- Top 5 (largest elec demand increase) ---")
print(df_demand_state[['state', 'elec_change_gwh', 'pct_elec_demand_change',
                        'site_energy_change_gwh', 'pct_site_energy_change']].head(5).to_string(index=False))
print(f"\n--- Bottom 5 ---")
print(df_demand_state[['state', 'elec_change_gwh', 'pct_elec_demand_change',
                        'site_energy_change_gwh', 'pct_site_energy_change']].tail(5).to_string(index=False))
print("\n✓ STEP Xb COMPLETE")

---
## Step X: Geospatial Visualization
---

In [ ]:
gdf_conus = None
gdf_alaska = None

try:
    gdf_states_raw = gpd.read_file(SHAPEFILE_PATH)
    _, gdf_conus, gdf_alaska = prepare_state_geodataframe(gdf_states_raw, df_spark, merge_col='state')
    print(f"✓ Geodataframe prepared: CONUS={len(gdf_conus)}, AK={len(gdf_alaska)}")
except Exception as e:
    print(f"⚠ Shapefile not loaded: {e} — skipping maps")

In [ ]:
# Demand change map (diverging)
if gdf_conus is not None and gdf_alaska is not None:
    _, gdf_demand_conus, gdf_demand_alaska = prepare_state_geodataframe(
        gdf_states_raw, df_demand_state, merge_col='state'
    )
    create_choropleth_map(
        gdf_demand_conus, gdf_demand_alaska,
        column='elec_change_gwh',
        title='Electricity Demand Change Under 100% HP Adoption by State (2022)',
        cbar_label='Electricity Demand Change (GWh)\n(positive = more grid electricity needed)',
        output_path=os.path.join(PROJECT_ROOT, "state_elec_demand_change_map_2022.png"),
        cmap='coolwarm', show_plot=True,
    )
    print("✓ Demand map generated")
else:
    print("⚠ Maps skipped")

---
## Display Results
---

In [ ]:
# ============================================================================
# DISPLAY: DEMAND CHANGE
# ============================================================================

print(f"\n===== DEMAND CHANGE (MP{primary_mp}, GWh, all fuels, 100% adoption) =====\n")
display(df_demand_state[['state', 'home_count', 'elec_change_gwh',
                          'pct_elec_demand_change', 'site_energy_change_gwh',
                          'pct_site_energy_change']])

print(f"\n✓ DISPLAY COMPLETE")